In [1]:
import pandas as pd
import numpy as np
import hnswlib as hl
import hnsw_benchmark as bm
import twonn
import farhest_point_order as fp
import matplotlib.pyplot as plt
import os

In [9]:
# Alapadatok megadása
dataset_names = ["arguana"]
embedding_models = [
    "e5-small-v2",
    "multilingual-e5-large",
    "bge-small-en",
    "stella-base-en-v2",
    "all-MiniLM-L6-v2"
]
options = [
    "_lid_fps_lpca_2nn",
    "_lid_fps",
    "_lid",
    "_lid",
    "_lid"
]

df_corpus = {}
for dataset_name in dataset_names:
    for embedding_model, option in zip(embedding_models, options):
        corpus_filename = dataset_name + "_corpus_" + embedding_model + option
        corpus_input_path = os.path.join("..\\data\\", corpus_filename + ".json")
        corpus_data = pd.read_json(corpus_input_path, orient="records", lines=True)
        df_corpus[dataset_name + "_" + embedding_model] = corpus_data

df_query = {}
for dataset_name in dataset_names:
    for embedding_model in embedding_models:
        query_filename = dataset_name + "_query_" + embedding_model
        query_input_path = os.path.join("..\\data\\", query_filename + ".json")
        query_data = pd.read_json(query_input_path, orient="records", lines=True)
        df_query[dataset_name + "_" + embedding_model] = query_data

In [11]:
for name in df_corpus:
    print("-- " + name + " --")

    print("Corpus data")
    corpus_vectors = np.asarray([np.asarray(v) for v in df_corpus[name].vectors], dtype=np.float32)
    print(corpus_vectors.shape)
    print(df_corpus[name].keys())

    print("Query data")
    query_vectors = np.asarray([np.asarray(v) for v in df_query[name].vectors], dtype=np.float32)
    print(query_vectors.shape)

    print()


-- arguana_e5-small-v2 --
Corpus data
(8674, 384)
Index(['vectors', 'LID', 'fp_score', 'LPCA_dim', 'TwoNN'], dtype='str')
Query data
(1406, 384)

-- arguana_multilingual-e5-large --
Corpus data
(8674, 1024)
Index(['vectors', 'LID', 'fp_score'], dtype='str')
Query data
(1406, 1024)

-- arguana_bge-small-en --
Corpus data
(8674, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 384)

-- arguana_stella-base-en-v2 --
Corpus data
(8674, 768)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 768)

-- arguana_all-MiniLM-L6-v2 --
Corpus data
(8674, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 384)



In [12]:
# Benchmarkok futtatása
def run_benchmark(corpus_data, sorted_indices, query_data, n_times=1, ef_search=10):
    recalls = []
    for _ in range(0, n_times):
        bm_data = [corpus_data[i] for i in sorted_indices]
        benchmark = bm.HNSW_benchmark(data=bm_data)
        recall_list = benchmark.run(qs=query_data, ef_search=ef_search)
        recalls.append(np.mean(recall_list))

    print(f"Minimum recall: {np.min(recalls):.4f}")
    print(f"Average recall: {np.mean(recalls):.4f}")
    print(f"Maximum recall: {np.max(recalls):.4f}")

    # plt.figure(figsize=(8, 5))
    # plt.hist(recalls, bins=10, edgecolor="black", alpha=0.75)
    # plt.title("Distribution of recall values")
    # plt.xlabel("Recall")
    # plt.ylabel("Count")
    # plt.grid(axis="y", alpha=0.3)
    # plt.ylim(0, len(df_query.vectors))
    # plt.show()

In [13]:
# HNSW benchmark - alap beállítások
n_times = 100
ef_search = 10

In [14]:
# HNSW benchmark - Random beszúrás
print("-- Random --")
print()
print()
for name in df_corpus:
    print("-- " + name + " --")

    sorted_indices = np.arange(len(df_corpus[name].vectors))
    run_benchmark(df_corpus[name].vectors, sorted_indices, df_query[name].vectors, n_times, ef_search)

    print()

-- Random --


-- arguana_e5-small-v2 --
Minimum recall: 0.9055
Average recall: 0.9135
Maximum recall: 0.9283

-- arguana_multilingual-e5-large --
Minimum recall: 0.9058
Average recall: 0.9134
Maximum recall: 0.9213

-- arguana_bge-small-en --
Minimum recall: 0.9131
Average recall: 0.9195
Maximum recall: 0.9267

-- arguana_stella-base-en-v2 --
Minimum recall: 0.9096
Average recall: 0.9152
Maximum recall: 0.9229

-- arguana_all-MiniLM-L6-v2 --
Minimum recall: 0.9500
Average recall: 0.9562
Maximum recall: 0.9633



In [15]:
# HNSW benchmark - LID szerint csökkenő
print("-- Descending by LID --")
print()
print()
for name in df_corpus:
    if "LID" in df_corpus["arguana_multilingual-e5-large"].keys():
        print("-- " + name + " --")

        sorted_indices = np.argsort(df_corpus[name].LID, descending=True)
        run_benchmark(df_corpus[name].vectors, sorted_indices, df_query[name].vectors, n_times, ef_search)

        print()

-- Descending by LID --


-- arguana_e5-small-v2 --
Minimum recall: 0.9466
Average recall: 0.9498
Maximum recall: 0.9548

-- arguana_multilingual-e5-large --
Minimum recall: 0.9546
Average recall: 0.9569
Maximum recall: 0.9615

-- arguana_bge-small-en --
Minimum recall: 0.9617
Average recall: 0.9656
Maximum recall: 0.9681

-- arguana_stella-base-en-v2 --
Minimum recall: 0.9578
Average recall: 0.9606
Maximum recall: 0.9634

-- arguana_all-MiniLM-L6-v2 --
Minimum recall: 0.9858
Average recall: 0.9876
Maximum recall: 0.9900



In [16]:
# HNSW benchmark - Farthes point score szerint csökkenő
print("-- Descending by farthes point score --")
print()
print()
for name in df_corpus:
    if "fp_score" in df_corpus[name].keys():
        print("-- " + name + " --")

        sorted_indices = np.argsort(df_corpus[name].fp_score, descending=True)
        run_benchmark(df_corpus[name].vectors, sorted_indices, df_query[name].vectors, n_times, ef_search)

        print()

-- Descending by farthes point score --


-- arguana_e5-small-v2 --
Minimum recall: 0.9514
Average recall: 0.9544
Maximum recall: 0.9568

-- arguana_multilingual-e5-large --
Minimum recall: 0.9587
Average recall: 0.9621
Maximum recall: 0.9642

